In [1]:
import re
import subprocess
from typing import List, Tuple
from rdflib import Graph
import requests
import json

# Config
TTL_FILE = "movie_kb_final.ttl"
OLLAMA_URL = "http://localhost:11434/api/generate"
GEMMA_MODEL = "gemma:2b"
MAX_PREDICATES = 80
MAX_CLASSES = 40
SAMPLE_TRIPLES = 20
OLLAMA_CMD = ["ollama", "run"]

# Call LLM
def ask_local_llm(prompt: str, model: str=GEMMA_MODEL):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    response = requests.post(OLLAMA_URL, json=payload)
    if response.status_code != 200:
        raise RuntimeError(f"Ollama API error {response.status_code}: {response.text}")
    data = response.json()
    return data.get("response", "")

# Test
print(ask_local_llm("Say hello in one sentence."))

/Users/hanwang/Desktop/4下/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Hello! How can I assist you today?


 1) Load RDF graph

In [2]:
# RDF pic
def load_graph(ttl_path):
    g = Graph()
    g.parse(ttl_path, format="turtle")
    print(f"Loaded {len(g)} triples from {ttl_path}")
    return g

2) Build a small schema summary

In [3]:
def get_prefix_block(g: Graph) -> str:
    defaults = {
        "rdf":  "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
        "xsd":  "http://www.w3.org/2001/XMLSchema#",
        "owl":  "http://www.w3.org/2002/07/owl#",
    }
    ns_map = {p: str(ns) for p, ns in g.namespace_manager.namespaces()}
    for k, v in defaults.items():
        ns_map.setdefault(k, v)
    lines = [f"PREFIX {p}: <{ns}>" for p, ns in ns_map.items()]
    return "\n".join(sorted(lines))

# Query all predicates
def list_distinct_predicates(g, limit=MAX_PREDICATES):
    q = f"SELECT DISTINCT ?p WHERE {{ ?s ?p ?o . }} LIMIT {limit}"
    return [str(row.p) for row in g.query(q)]

# List all classes
def list_distinct_classes(g, limit=MAX_CLASSES):
    q = f"SELECT DISTINCT ?cls WHERE {{ ?s a ?cls . }} LIMIT {limit}"
    return [str(row.cls) for row in g.query(q)]

# Sample Triplet
def sample_triples(g, limit=SAMPLE_TRIPLES):
    q = f"SELECT ?s ?p ?o WHERE {{ ?s ?p ?o . }} LIMIT {limit}"
    return [(str(r.s), str(r.p), str(r.o)) for r in g.query(q)]

def build_schema_summary(g: Graph) -> str:
    prefixes = get_prefix_block(g)
    preds    = list_distinct_predicates(g)
    clss     = list_distinct_classes(g)
    samples  = sample_triples(g)

    pred_lines   = "\n".join(f"- {p}" for p in preds)
    cls_lines    = "\n".join(f"- {c}" for c in clss)
    sample_lines = "\n".join(f"- {s} {p} {o}" for s, p, o in samples)

    summary = f"""
{prefixes}

# Predicates (sampled, unique up to {MAX_PREDICATES})
{pred_lines}

# Classes / rdf:type (sampled, unique up to {MAX_CLASSES})
{cls_lines}

# Sample triples (up to {SAMPLE_TRIPLES})
{sample_lines}
"""
    return summary.strip()
# Test
g = load_graph(TTL_FILE)
schema = build_schema_summary(g)
print(schema[:500])

Loaded 61809 triples from movie_kb_final.ttl
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX csvw: <http://www.w3.org/ns/csvw#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcam: <http://purl.org/dc/dcam/>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dcmitype: <http://purl.org/dc/dcmitype/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX doap: <http://usefulinc.com/ns/doap#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX mv: <http://moviekg.local/resource/>
P


3) Prompting Gemma: NL → SPARQL

In [4]:
SPARQL_INSTRUCTIONS = """
You are a SPARQL generator. Convert the user QUESTION into a valid SPARQL
1.1 SELECT query for the given RDF graph schema. Follow strictly:
- Use ONLY the IRIs/prefixes visible in the SCHEMA SUMMARY.
- Prefer readable SELECT projections with variable names.
- Do NOT invent new predicates/classes.
- Return ONLY the SPARQL query in a single fenced code block labeled ```sparql
- No explanations or extra text outside the code block.
"""

def make_sparql_prompt(schema_summary: str, question: str) -> str:
    return f"""{SPARQL_INSTRUCTIONS}
SCHEMA SUMMARY:
{schema_summary}

QUESTION:
{question}

Return only the SPARQL query in a code block.
"""

CODE_BLOCK_RE = re.compile(r"```(?:sparql)?\s*(.*?)```", re.IGNORECASE | re.DOTALL)

def extract_sparql_from_text(text: str) -> str:
    m = CODE_BLOCK_RE.search(text)
    if m:
        return m.group(1).strip()
    return text.strip()

def generate_sparql(question: str, schema_summary: str) -> str:
    raw   = ask_local_llm(make_sparql_prompt(schema_summary, question))
    query = extract_sparql_from_text(raw)
    return query

# Note: Since the SPARQL generated by gemma:2b is inconsistent in quality, we have replaced `generate_sparql` with template matching.

TEMPLATES = {
    "movies_by_director": """
        PREFIX mvo: <http://moviekg.local/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT ?movie ?label WHERE {{
            ?movie mvo:hasDirector ?dir .
            ?dir rdfs:label "{name}" .
            OPTIONAL {{ ?movie rdfs:label ?label }}
        }}""",
    "movies_by_actor": """
        PREFIX mvo: <http://moviekg.local/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT ?movie ?label WHERE {{
            ?movie mvo:hasActor ?actor .
            ?actor rdfs:label "{name}" .
            OPTIONAL {{ ?movie rdfs:label ?label }}
        }}""",
    "movies_by_genre": "",
    "director_of_movie": """
        PREFIX mvo: <http://moviekg.local/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT ?director ?label WHERE {{
            ?movie rdfs:label ?title .
            FILTER(LCASE(STR(?title)) = LCASE("{name}"))
            ?movie mvo:hasDirector ?director .
            OPTIONAL {{ ?director rdfs:label ?label }}
        }}"""
}

def build_genre_map(g: Graph) -> dict:
    query = """
        PREFIX mvo: <http://moviekg.local/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT ?genre ?label WHERE {
            ?s mvo:hasGenre ?genre .
            OPTIONAL { ?genre rdfs:label ?label }
        }
    """
    vars_, rows = run_sparql(g, query)
    genre_map = {}
    for genre_uri, label in rows:
        if label and label != "None":
            genre_map[label.lower()] = genre_uri
    return genre_map

def classify_question(question: str):
    prompt = f"""Classify this question into one of these types and extract the key name.

TYPES:
- movies_by_director: asking for movies made by a director
- movies_by_actor: asking for movies with an actor
- movies_by_genre: asking for movies of a genre
- director_of_movie: asking who directed a specific movie

Reply in this exact format:
TYPE: [one of the types above]
NAME: [the name extracted from the question]

Examples:
Q: What movies has Christopher Nolan directed? -> TYPE: movies_by_director, NAME: Christopher Nolan
Q: Who directed Inception? -> TYPE: director_of_movie, NAME: Inception
Q: What movies did Tom Hanks act in? -> TYPE: movies_by_actor, NAME: Tom Hanks
Q: What are some Action films? -> TYPE: movies_by_genre, NAME: Action

Question: {question}"""

    response = ask_local_llm(prompt)
    q_type, name = None, None
    for line in response.strip().split("\n"):
        line = line.strip()
        if line.startswith("TYPE:"):
            q_type = line[5:].strip().strip("[]")
        elif line.startswith("NAME:"):
            name = line[5:].strip().strip("[]")
    return q_type, name

4) Execute SPARQL with rdflib (and optional self-repair)

In [6]:
def run_sparql(g: Graph, query: str) -> Tuple[List[str], List[Tuple]]:
 res = g.query(query)
 vars_ = [str(v) for v in res.vars]
 rows = [tuple(str(cell) for cell in r) for r in res]
 return vars_, rows

REPAIR_INSTRUCTIONS = """
The previous SPARQL failed to execute. Using the SCHEMA SUMMARY and the
ERROR MESSAGE,
return a corrected SPARQL 1.1 SELECT query. Follow strictly:
- Use only known prefixes/IRIs.
- Keep it as simple and robust as possible.
- Return only a single code block with the corrected SPARQL.
"""

def repair_sparql(schema_summary: str, question: str, bad_query: str,
error_msg: str) -> str:
    prompt = f"""{REPAIR_INSTRUCTIONS}
SCHEMA SUMMARY:
{schema_summary}
ORIGINAL QUESTION:
{question}
BAD SPARQL:
{bad_query}
ERROR MESSAGE:
{error_msg}
Return only the corrected SPARQL in a code block.
"""
    raw = ask_local_llm(prompt)
    return extract_sparql_from_text(raw)

 5) Orchestration: Ask with SPARQL-generation RAG

In [7]:
def answer_with_sparql_generation(g: Graph, schema_summary: str, question: str,
                                   try_repair: bool = True) -> dict:
    sparql = generate_sparql(question, schema_summary)
    try:
        vars_, rows = run_sparql(g, sparql)
        return {"query": sparql, "vars": vars_, "rows": rows, "repaired": False, "error": None}
    except Exception as e:
        err = str(e)
        if try_repair:
            repaired = repair_sparql(schema_summary, question, sparql, err)
            try:
                vars_, rows = run_sparql(g, repaired)
                return {"query": repaired, "vars": vars_, "rows": rows, "repaired": True, "error": None}
            except Exception as e2:
                return {"query": repaired, "vars": [], "rows": [], "repaired": True, "error": str(e2)}
        else:
            return {"query": sparql, "vars": [], "rows": [], "repaired": False, "error": err}

# Note: `answer_with_sparql_generation` depends on `generate_sparql`, but since gemma:2b is unstable,
# we have added `answer_with_template` as an alternative, which uses template matching combined with LLM classification.

def answer_with_template(g: Graph, genre_map: dict, question: str) -> dict:
    q_type, name = classify_question(question)

    if not q_type or not name:
        return {"query": "", "error": "Could not classify question",
                "rows": [], "vars": [], "repaired": False}

    if q_type == "movies_by_genre":
        name_lower = name.lower()
        genre_uri  = None
        for label, uri in genre_map.items():
            if name_lower in label or label in name_lower:
                genre_uri = uri
                break
        if not genre_uri:
            return {"query": "", "error": f"Genre '{name}' not found",
                    "rows": [], "vars": [], "repaired": False}
        query = f"""
            PREFIX mvo: <http://moviekg.local/ontology/>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            SELECT ?movie ?label WHERE {{
                ?movie mvo:hasGenre <{genre_uri}> .
                OPTIONAL {{ ?movie rdfs:label ?label }}
            }} LIMIT 20
        """
    elif q_type in TEMPLATES:
        query = TEMPLATES[q_type].format(name=name)
    else:
        return {"query": "", "error": f"Unknown type: {q_type}",
                "rows": [], "vars": [], "repaired": False}

    try:
        vars_, rows = run_sparql(g, query)
        return {"query": query, "vars": vars_, "rows": rows,
                "error": None, "repaired": False}
    except Exception as e:
        return {"query": query, "vars": [], "rows": [],
                "error": str(e), "repaired": False}

 6) (Baseline) Direct LLM answer w/o KG

In [8]:
def answer_no_rag(question):
    prompt = f"Answer the following question as best as you can:\n\n{question}"
    return ask_local_llm(prompt)

 7) CLI demo

In [9]:
def pretty_print_result(result: dict):
    if result.get("error"):
        print("\n[Execution Error]", result["error"])
    print("\n[SPARQL Query Used]")
    print(result["query"])
    print("\n[Repaired?]", result["repaired"])
    vars_ = result.get("vars", [])
    rows = result.get("rows", [])
    if not rows:
        print("\n[No rows returned]")
        return
    print("\n[Results]")
    print(" | ".join(vars_))
    for r in rows[:20]:
        print(" | ".join(r))
    if len(rows) > 20:
        print(f"... (showing 20 of {len(rows)})")


In [11]:
g= load_graph(TTL_FILE)
genre_map = build_genre_map(g)
schema= build_schema_summary(g)

questions = [
    "What movies has Christopher Nolan directed?",
    "What movies did Tom Hanks act in?",
    "What are some Action films?",
    "Who directed Inception?",
    "What movies were produced by Warner Bros?"
]

for q in questions:
    print(f"Question: {q}")
    print("\n--- Baseline (No RAG) ---")
    print(answer_no_rag(q)[:300])
    print("\n--- Template RAG ---")
    result = answer_with_template(g, genre_map, q)
    pretty_print_result(result)

Loaded 61809 triples from movie_kb_final.ttl
Question: What movies has Christopher Nolan directed?

--- Baseline (No RAG) ---
Christopher Nolan has directed movies such as Memento, The Prestige, Inception, Dunkirk, Interstellar, and Tenet.

--- Template RAG ---

[SPARQL Query Used]

        PREFIX mvo: <http://moviekg.local/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT ?movie ?label WHERE {
            ?movie mvo:hasDirector ?dir .
            ?dir rdfs:label "Christopher Nolan" .
            OPTIONAL { ?movie rdfs:label ?label }
        }

[Repaired?] False

[Results]
movie | label
http://moviekg.local/resource/movie_27205 | None
http://moviekg.local/resource/movie_872585 | None
Question: What movies did Tom Hanks act in?

--- Baseline (No RAG) ---
Sure, here's a summary of Tom Hanks' movies:

* Forrest Gump (1994)
* Cast Away (2000)
* Sully (2010)
* Saving Private Ryan (1998)
* Apollo 13 (2015)
* The Da Vinci Code (2012)
* Sully: The Musical (